# Etape 5 — Validation croisee et recherche d'hyperparametres
## Projet House Prices — Estimation du prix de vente d'un logement

**Objectif de ce notebook :**
1. Remplacer le "premier apercu" de l'etape 4 (un seul split train/test)
   par une **validation croisee a 5 folds** : moyenne **et ecart-type**
   du R²/MAE/RMSE pour chaque modele et chaque cible -- la vraie mesure de
   fiabilite, pas un chiffre qui depend du hasard d'un seul split.
2. Trancher serieusement entre `SalePrice` brute et `log1p(SalePrice)`,
   sur la base de cette mesure robuste.
3. **`GridSearchCV`** pour chercher les meilleurs hyperparametres des 3
   modeles, avec un objectif explicite : reduire le surapprentissage
   repere a l'etape 4 (`RandomForest` en tete).

Rappel de vocabulaire (question posee a l'etape 4) : ici, contrairement au
tableau de l'etape 4, on calcule bien un **ecart-type** au sens
statistique -- la dispersion du R² mesure sur chacun des 5 folds, pas une
simple difference entre 2 nombres.

## 1. Import et chargement des donnees

In [1]:
import sys
import time
import numpy as np
import pandas as pd
import joblib
import os
from functools import partial

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, make_scorer

sys.path.append('..')
from src.preprocessing import run_pipeline, build_full_pipeline

RANDOM_STATE = 42

result = run_pipeline('../data/raw/House_Prices.csv')
X_train_raw = result['X_train_raw']
X_test_raw = result['X_test_raw']
y_train = result['y_train']
y_test = result['y_test']

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print('X_train_raw :', X_train_raw.shape)
print('X_test_raw  :', X_test_raw.shape)


X_train_raw : (1166, 87)
X_test_raw  : (292, 87)


## 2. Scorers "en dollars", valables pour les 2 cibles

In [2]:
# GridSearchCV et cross_validate ne connaissent que la cible qu'on leur donne
# (brute ou log1p) : si on veut comparer les deux sur la MEME echelle
# (des dollars), il faut un scorer qui sait re-transformer via expm1() avant
# de calculer la metrique -- sinon un R2 calcule en 'espace log' n'est pas
# comparable a un R2 calcule en 'espace dollars'.

def _r2_dollars(y_true, y_pred, is_log):
    if is_log:
        y_true, y_pred = np.expm1(y_true), np.expm1(y_pred)
    return r2_score(y_true, y_pred)

def _mae_dollars(y_true, y_pred, is_log):
    if is_log:
        y_true, y_pred = np.expm1(y_true), np.expm1(y_pred)
    return mean_absolute_error(y_true, y_pred)

def _rmse_dollars(y_true, y_pred, is_log):
    if is_log:
        y_true, y_pred = np.expm1(y_true), np.expm1(y_pred)
    return root_mean_squared_error(y_true, y_pred)

def make_scorers(is_log):
    return {
        'r2': make_scorer(partial(_r2_dollars, is_log=is_log)),
        'mae': make_scorer(partial(_mae_dollars, is_log=is_log), greater_is_better=False),
        'rmse': make_scorer(partial(_rmse_dollars, is_log=is_log), greater_is_better=False),
    }


## 3. Validation croisee (baseline, sans tuning) : 3 modeles x 2 cibles

In [3]:
models = {
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),  # meme choix que l'etape 4
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = []
for model_name, model in models.items():
    for target_name, y_tr, is_log in [('brute', y_train, False), ('log1p', y_train_log, True)]:
        pipeline = build_full_pipeline(model, X_train_raw)
        scorers = make_scorers(is_log)

        start = time.time()
        scores = cross_validate(pipeline, X_train_raw, y_tr, cv=kf, scoring=scorers, n_jobs=-1)
        elapsed = time.time() - start

        cv_results.append({
            'Modele': model_name,
            'Cible': target_name,
            'R2 CV (moyenne)': round(scores['test_r2'].mean(), 3),
            'R2 CV (ecart-type)': round(scores['test_r2'].std(), 3),
            'MAE CV (moyenne $)': round(-scores['test_mae'].mean(), 0),
            'RMSE CV (moyenne $)': round(-scores['test_rmse'].mean(), 0),
            'Temps CV (s)': round(elapsed, 1),
        })

cv_df = pd.DataFrame(cv_results).sort_values('R2 CV (moyenne)', ascending=False).reset_index(drop=True)
cv_df


,Modele,Cible,R2 CV (moyenne),R2 CV (ecart-type),MAE CV (moyenne $),RMSE CV (moyenne $),Temps CV (s)
0,Ridge,log1p,0.930,0.011,14096.0,20959.0,0.3
1,GradientBoosting,log1p,0.911,0.008,15161.0,23808.0,3.7
2,GradientBoosting,brute,0.911,0.011,15391.0,23857.0,3.6
3,RandomForest,brute,0.891,0.009,16609.0,26336.0,18.7
4,RandomForest,log1p,0.889,0.006,16595.0,26591.0,18.1
5,Ridge,brute,0.875,0.024,18843.0,27876.0,0.3


**Interpretation :** `R2 CV (moyenne)` est desormais accompagne d'un vrai
**ecart-type** (colonne `R2 CV (ecart-type)`) -- la dispersion du score
mesure sur les 5 folds. Un ecart-type faible signifie que le score ne
depend pas trop du decoupage train/test choisi ; un ecart-type eleve
signifierait un modele instable, sensible aux donnees precises qu'il voit.
Comparer ce tableau a celui de l'etape 4 (un seul split) permet de voir si
le classement des modeles tient toujours -- si un modele semblait bon sur
un split par chance, la CV le revele ici.

## 4. Trancher : cible brute ou `log1p` ?

In [4]:
target_comparison = cv_df.groupby('Cible')['R2 CV (moyenne)'].mean().sort_values(ascending=False)
print("R2 CV moyen (tous modeles confondus), par cible :")
print(target_comparison)

CHOSEN_TARGET = target_comparison.index[0]
print(f"\nCible retenue pour la suite (GridSearchCV) : {CHOSEN_TARGET}")


R2 CV moyen (tous modeles confondus), par cible :
Cible
log1p    0.910000
brute    0.892333
Name: R2 CV (moyenne), dtype: float64

Cible retenue pour la suite (GridSearchCV) : log1p


**Interpretation :** on tranche sur la base de la moyenne des R² CV des 3
modeles (une decision unique pour tous les modeles, pour garder le
notebook lisible, plutot qu'une cible differente par modele). Si l'ecart
entre `brute` et `log1p` s'avere faible, ce choix a peu d'impact pratique
-- ce qui compte surtout a ce stade, c'est le tuning qui suit.

## 5. `GridSearchCV` : recherche d'hyperparametres pour chaque modele

In [5]:
y_target = y_train_log if CHOSEN_TARGET == 'log1p' else y_train
scoring = make_scorers(is_log=(CHOSEN_TARGET == 'log1p'))['r2']

param_grids = {
    'Ridge': {
        'estimator': Ridge(random_state=RANDOM_STATE),
        'param_grid': {'model__alpha': [0.1, 1, 10, 50, 100]},
    },
    'RandomForest': {
        'estimator': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
        'param_grid': {
            'model__max_depth': [10, 15, None],
            'model__min_samples_leaf': [1, 3],
        },
    },
    'GradientBoosting': {
        'estimator': GradientBoostingRegressor(n_estimators=150, random_state=RANDOM_STATE),
        'param_grid': {
            'model__learning_rate': [0.05, 0.1],
            'model__max_depth': [2, 3],
            'model__subsample': [0.8, 1.0],
        },
    },
}

grid_results = {}

for model_name, cfg in param_grids.items():
    pipeline = build_full_pipeline(cfg['estimator'], X_train_raw)
    grid = GridSearchCV(
        pipeline,
        param_grid=cfg['param_grid'],
        scoring=scoring,
        cv=kf,
        n_jobs=-1,
        refit=True,
    )
    start = time.time()
    grid.fit(X_train_raw, y_target)
    elapsed = time.time() - start

    grid_results[model_name] = grid
    print(f"{model_name:20s} -> meilleurs params : {grid.best_params_}")
    print(f"{'':20s}    R2 CV (best)   : {grid.best_score_:.3f}")
    print(f"{'':20s}    temps GridSearch : {elapsed:.1f}s\n")


Ridge                -> meilleurs params : {'model__alpha': 10}
                        R2 CV (best)   : 0.935
                        temps GridSearch : 1.3s



RandomForest         -> meilleurs params : {'model__max_depth': None, 'model__min_samples_leaf': 1}
                        R2 CV (best)   : 0.889
                        temps GridSearch : 93.3s



GradientBoosting     -> meilleurs params : {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__subsample': 1.0}
                        R2 CV (best)   : 0.912
                        temps GridSearch : 35.3s



**Interpretation :** resultats a nuancer, pas a enjoliver.
- **`Ridge` progresse nettement** : `alpha=10` donne un
  R² CV de 0.935, mieux que `alpha=1.0` par defaut (0.930 en section 3) --
  la regularisation aide un peu, comme attendu vu la colinearite deja
  identifiee a l'etape 2.
- **`GradientBoosting` progresse legerement** (0.912 vs 0.911 en
  baseline) : le tuning fin (`learning_rate`, `max_depth`, `subsample`)
  n'apporte quasiment rien ici -- les valeurs par defaut etaient deja
  proches de l'optimum sur cette grille.
- **`RandomForest` ne progresse PAS** : le meilleur combo retenu
  (`max_depth=None`, `min_samples_leaf=1`) est... la configuration la
  moins regularisee de toute la grille, essentiellement equivalente aux
  reglages par defaut. Son R² CV (0.889) est identique a la baseline de
  la section 3. Autrement dit, sur CETTE grille, aucune des options plus
  regularisees testees (`max_depth` limite, `min_samples_leaf=3`) n'a fait
  mieux -- le surapprentissage de `RandomForest` n'est donc pas resolu ici.
  Une grille plus large (`max_depth` encore plus limite, `max_features`,
  `min_samples_split`) ou un autre modele pourrait etre necessaire pour
  vraiment le corriger -- a garder en tete pour l'etape 6.

## 6. Evaluation finale sur le test, avec les modeles tunes

In [6]:
final_results = []

for model_name, grid in grid_results.items():
    best_pipeline = grid.best_estimator_

    pred_train = best_pipeline.predict(X_train_raw)
    pred_test = best_pipeline.predict(X_test_raw)

    if CHOSEN_TARGET == 'log1p':
        pred_train_dollars = np.expm1(pred_train)
        pred_test_dollars = np.expm1(pred_test)
    else:
        pred_train_dollars = pred_train
        pred_test_dollars = pred_test

    final_results.append({
        'Modele': model_name,
        'Meilleurs params': grid.best_params_,
        'R2 CV (best)': round(grid.best_score_, 3),
        'R2 train': round(r2_score(y_train, pred_train_dollars), 3),
        'R2 test': round(r2_score(y_test, pred_test_dollars), 3),
        'MAE test ($)': round(mean_absolute_error(y_test, pred_test_dollars), 0),
        'RMSE test ($)': round(root_mean_squared_error(y_test, pred_test_dollars), 0),
    })

final_df = pd.DataFrame(final_results).sort_values('R2 test', ascending=False).reset_index(drop=True)
final_df


,Modele,Meilleurs params,R2 CV (best),R2 train,R2 test,MAE test ($),RMSE test ($)
0,Ridge,{'model__alpha': 10},0.935,0.954,0.929,14214.0,19834.0
1,GradientBoosting,"{'model__learning_rate': 0.1, 'model__max_dept...",0.912,0.979,0.925,14207.0,20368.0
2,RandomForest,"{'model__max_depth': None, 'model__min_samples...",0.889,0.984,0.906,15758.0,22800.0


**Interpretation :** le classement change par rapport a l'etape 4 --
**`Ridge` devient le meilleur modele**
(R² test 0.929), devant `GradientBoosting` (0.925) et `RandomForest`
(0.906). C'est coherent avec la section precedente : `Ridge` est le seul
des 3 a avoir reellement beneficie du tuning. Le R² CV et le R² test
restent proches pour les 3 modeles (ecart <= 0.02), signe que la mesure
CV est fiable -- pas de mauvaise surprise sur le test.

Point a ne pas balayer : l'ecart `R2 train`/`R2 test` de `RandomForest`
reste le plus large des 3 (0.984 vs 0.906, ecart 0.078) -- le
surapprentissage est attenue par rapport a l'etape 4, mais pas elimine,
et le tuning fait ici n'y change rien (cf. section precedente). Sur ce
dataset et avec ces 3 modeles, **le modele le plus simple (regression
lineaire regularisee) l'emporte** -- un resultat a assumer pour l'etape 6
plutot qu'a masquer, plutot que de partir du principe qu'un modele plus
complexe est necessairement meilleur.

## 7. Sauvegarde des pipelines tunes

In [7]:
os.makedirs('../models', exist_ok=True)

for model_name, grid in grid_results.items():
    path = f'../models/{model_name}_tuned.joblib'
    joblib.dump(grid.best_estimator_, path)
    print(f'Sauvegarde : {path}')

print(f"\nCible retenue pour ces modeles tunes : {CHOSEN_TARGET}")


Sauvegarde : ../models/Ridge_tuned.joblib
Sauvegarde : ../models/RandomForest_tuned.joblib
Sauvegarde : ../models/GradientBoosting_tuned.joblib

Cible retenue pour ces modeles tunes : log1p


## Conclusion de l'etape 5

- La validation croisee (5 folds) remplace le "premier apercu" sur un
  seul split de l'etape 4 : chaque score est desormais accompagne d'un
  **ecart-type**, la vraie mesure de sa fiabilite.
- La cible (`brute` ou `log1p`) a ete tranchee sur la base de cette
  mesure robuste, pas d'un chiffre isole.
- `GridSearchCV` a cherche les meilleurs hyperparametres pour les 3
  modeles, avec un objectif explicite de reduire le surapprentissage de
  `RandomForest` identifie a l'etape 4.
- 3 pipelines **tunes** sont sauvegardes dans `models/*_tuned.joblib`,
  prets pour l'evaluation comparative formelle.

**Prochaine etape (etape 6) :** evaluation comparative formelle des
modeles tunes (MAE/RMSE/R² definitifs, analyse des residus), pour
designer LE modele final qui sera utilise a l'etape 7 (interpretation)
puis integre a l'app Streamlit (etape 8).